# Stage S05 — Urban Microclimate Feature Engineering

**Project:** Urban Microclimate Modelling via Neural Surrogates and CFD  
**Repository:** [mahir-m01/Urban-Microclimate-Modelling](https://github.com/mahir-m01/Urban-Microclimate-Modelling)  
**Status at notebook start:** EDA ✅ · Preprocessing ✅ · Feature-Engineering modules ✅ · **This notebook: execute & validate**

---

## What this notebook does

This notebook **does not** redo EDA or preprocessing. It:

1. Validates that the preprocessed inputs (building-height rasters `*_topo`, pedestrian-wind fields `*_ped.nc`, and case metadata) are present and well-formed.
2. Imports the **existing `.py` feature-engineering modules** from `pipeline/S05_feature_engineering/` — no logic is duplicated here.
3. Runs those modules to produce:
   - **Case-level feature table** (`case_features.csv`) — one row per simulation case, 70+ scalar features.
   - **22-channel pixel-level spatial tensors** (`pixel_features/<case>_features.npz`) — aligned with the CFD grid.
4. Validates every output for physical plausibility, absence of NaN/Inf, and zero target leakage.
5. Prints a final gate report.

## Feature groups produced

| Group | Module | Features |
|---|---|---|
| Domain geometry | `validation.py` | `nx`, `ny`, `dx`, `lambda_p`, … |
| Height statistics | `morphology.py` | `mean_h`, `std_h`, `max_h`, `cv_h`, `skew_h`, … |
| Frontal area & wall index | `orientation_features.py` | `lambda_f_x/y/dir`, `lambda_w`, `windward_wall_share`, … |
| Aerodynamic parameters | `aerodynamic.py` | `d_macdonald_m`, `z0_macdonald_m`, `d_kanda_m`, `d_over_hmax_kanda`, `z0_kanda_m`, `z0_over_hmax_kanda`, … |
| Distance to building | `spatial_features.py` | `dist_mean_m`, `dist_std_m`, `dist_max_m`, … (exact 3×3 periodic tiling) |
| Multi-scale local density & height | `spatial_features.py` | `lambda_p_{8,16,32,64}m`, `h_mean_{8,16,32,64}m` (pixel-level) |
| Directional fetch | `fetch_features.py` | `upwind_fetch_mean_m`, `downwind_fetch_mean_m`, `lateral_fetch_mean_m`, `upwind_h_{1_10m,11_30m,31_60m}` |
| 22-channel spatial tensor | `feature_assembly.py` | Channels 0–21 per pixel (see Section 5) |

## Expected inputs / outputs

| Input | Location |
|---|---|
| Building-height rasters | `data/raw/<case>/<case>_topo` |
| Pedestrian wind fields | `data/raw/<case>/<case>_ped.nc` |
| Master case table | `pipeline/S04_eda/S04g_combined_report/master_case_table.csv` |
| Model-ready case list | `pipeline/S04_eda/S04g_combined_report/model_ready_cases.csv` |

| Output | Location |
|---|---|
| Case-level features | `pipeline/S05_feature_engineering/outputs/case_features.csv` |
| Pixel tensors | `pipeline/S05_feature_engineering/outputs/pixel_features/<case>_features.npz` |
| Feature dictionary | `pipeline/S05_feature_engineering/outputs/feature_dictionary.json` |
| Gate report | `pipeline/S05_feature_engineering/outputs/S05_gate_report.md` |


---
## Section 2 — Environment Setup

In [ ]:
# ── Google Colab: clone the repository if needed ────────────────────────────
import os, sys

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    # Clone only once
    if not os.path.isdir('/content/Urban-Microclimate-Modelling'):
        !git clone https://github.com/mahir-m01/Urban-Microclimate-Modelling.git /content/Urban-Microclimate-Modelling
    os.chdir('/content/Urban-Microclimate-Modelling')
    print("✅ Working directory:", os.getcwd())
else:
    # Local: repo root is two levels above this notebook
    NOTEBOOK_DIR = os.path.abspath(os.path.dirname('__file__') if '__file__' in dir() else '.')
    REPO_ROOT = os.path.abspath(os.path.join(NOTEBOOK_DIR, '..'))
    os.chdir(REPO_ROOT)
    print("✅ Working directory:", os.getcwd())

In [ ]:
# ── Install missing dependencies (Colab-safe; skipped if already present) ───
import importlib, subprocess, sys

REQUIRED = {
    'numpy':   'numpy',
    'pandas':  'pandas',
    'scipy':   'scipy',
    'h5py':    'h5py',
    'matplotlib': 'matplotlib',
}

missing = [pip for mod, pip in REQUIRED.items() if importlib.util.find_spec(mod) is None]
if missing:
    print(f"Installing: {missing}")
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet'] + missing)
else:
    print("✅ All required packages already installed.")

In [ ]:
# ── Standard library imports ─────────────────────────────────────────────────
import json
import math
import os
import pathlib
import sys
import time
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import h5py

print(f"numpy  {np.__version__}")
print(f"pandas {pd.__version__}")

In [ ]:
# ── Path configuration ────────────────────────────────────────────────────────
# If running locally and your preprocessed data lives somewhere else,
# set REPO_ROOT to the cloned repository root.
# On Colab with a cloned repo, the default below is correct.

REPO_ROOT = pathlib.Path(os.getcwd()).resolve()

# ════════════════════════════════════════════════════════════════
# CONFIGURABLE: if your raw data lives outside the repository,
# override RAW_DATA_DIR here:
RAW_DATA_DIR_OVERRIDE = None   # e.g. pathlib.Path('/content/drive/MyDrive/urbantales/raw')
# ════════════════════════════════════════════════════════════════

DATA_DIR        = REPO_ROOT / 'data'
RAW_DATA_DIR    = pathlib.Path(RAW_DATA_DIR_OVERRIDE) if RAW_DATA_DIR_OVERRIDE else DATA_DIR / 'raw'
S04_REPORT_DIR  = REPO_ROOT / 'pipeline' / 'S04_eda' / 'S04g_combined_report'
S05_DIR         = REPO_ROOT / 'pipeline' / 'S05_feature_engineering'
S05_OUT_DIR     = S05_DIR / 'outputs'
PIXEL_OUT_DIR   = S05_OUT_DIR / 'pixel_features'

# Ensure repo root is importable
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print("REPO_ROOT     :", REPO_ROOT)
print("RAW_DATA_DIR  :", RAW_DATA_DIR)
print("S05_OUT_DIR   :", S05_OUT_DIR)
print("raw dir exists:", RAW_DATA_DIR.exists())

---
## Section 3 — Input Validation

We discover the available cases from the preprocessing outputs and check that every
required file is present before running anything expensive.

In [ ]:
# ── Load master case table and model-ready list ───────────────────────────────
MASTER_CSV      = S04_REPORT_DIR / 'master_case_table.csv'
MODEL_READY_CSV = S04_REPORT_DIR / 'model_ready_cases.csv'

for p in [MASTER_CSV, MODEL_READY_CSV]:
    if not p.exists():
        raise FileNotFoundError(
            f"\n❌ Required EDA output not found: {p}\n"
            "   This file is produced by Stage S04. Re-run S04 or pull the latest"
            " repository commit before proceeding."
        )

master_df = pd.read_csv(MASTER_CSV)
ready_df  = pd.read_csv(MODEL_READY_CSV)

print(f"Master case table : {len(master_df):>4} rows  |  cols: {list(master_df.columns[:8])} …")
print(f"Model-ready cases : {len(ready_df):>4} rows")

In [ ]:
# ── Discover which cases have local topo + ped files ─────────────────────────
def _has_topo_and_ped(case_id: str) -> dict:
    """Return dict with topo_ok, ped_ok paths for a given case."""
    topo = RAW_DATA_DIR / case_id / f"{case_id}_topo"
    ped  = RAW_DATA_DIR / case_id / f"{case_id}_ped.nc"
    return {'topo': topo, 'ped': ped, 'topo_ok': topo.exists(), 'ped_ok': ped.exists()}

all_cases = ready_df['case'].tolist()
file_status = {c: _has_topo_and_ped(c) for c in all_cases}

available_cases = [c for c, s in file_status.items() if s['topo_ok']]
missing_topo    = [c for c, s in file_status.items() if not s['topo_ok']]
missing_ped     = [c for c, s in file_status.items() if s['topo_ok'] and not s['ped_ok']]

print(f"Total model-ready cases   : {len(all_cases)}")
print(f"Cases with local topo file: {len(available_cases)}")
print(f"Cases missing topo        : {len(missing_topo)}")
print(f"Cases missing ped.nc      : {len(missing_ped)} (target Y will not be stored for these)")

if not available_cases:
    raise RuntimeError(
        "\n❌ No _topo files found in: " + str(RAW_DATA_DIR) +
        "\n   Either set RAW_DATA_DIR_OVERRIDE in Section 2 to point at your data,"
        " or download at least one case from the UrbanTales portal."
    )

print("\nAvailable cases:")
for c in available_cases:
    s = file_status[c]
    ped_note = "✅ ped.nc" if s['ped_ok'] else "⚠️  no ped.nc (target Y unavailable)"
    print(f"  {c:30s}  topo ✅  {ped_note}")

if missing_topo:
    print(f"\n⚠️  {len(missing_topo)} cases have no local topo — feature extraction will be skipped for them.")

In [ ]:
# ── Validate topo dimensions and value ranges ─────────────────────────────────
print("Validating topo files …\n")
topo_checks = []
for case_id in available_cases:
    row  = ready_df.loc[ready_df['case'] == case_id].iloc[0]
    topo = np.loadtxt(file_status[case_id]['topo'])
    expected_ny = int(row['ny'])
    expected_nx = int(row['nx'])
    dx          = float(row['dx'])
    dy          = float(row['dy'])
    h_min, h_max = topo.min(), topo.max()
    n_bldg = int((topo > 0).sum())
    lp_computed = n_bldg / topo.size
    shape_ok = (topo.shape == (expected_ny, expected_nx))
    finite_ok = bool(np.all(np.isfinite(topo)))
    nonneg_ok = bool(h_min >= 0.0)

    topo_checks.append({
        'case': case_id,
        'shape': topo.shape,
        'expected_shape': (expected_ny, expected_nx),
        'shape_ok': shape_ok,
        'dx_m': dx, 'dy_m': dy,
        'h_min_m': round(h_min, 3),
        'h_max_m': round(h_max, 3),
        'n_bldg_cells': n_bldg,
        'lambda_p': round(lp_computed, 5),
        'all_finite': finite_ok,
        'non_negative': nonneg_ok,
        'PASS': shape_ok and finite_ok and nonneg_ok,
    })

topo_df = pd.DataFrame(topo_checks)
display(topo_df[['case','shape','shape_ok','dx_m','h_min_m','h_max_m','n_bldg_cells','lambda_p','all_finite','non_negative','PASS']])

if not topo_df['PASS'].all():
    raise ValueError("❌ One or more topo files failed validation. Fix before proceeding.")
print("\n✅ All topo files pass validation.")

In [ ]:
# ── Validate metadata availability for available cases ───────────────────────
REQUIRED_META_COLS = [
    'case', 'subset', 'split_group', 'recommended_split',
    'nx', 'ny', 'dx', 'dy', 'forcing_dir', 'u_tau_metadata', 'roughness_length',
]
missing_cols = [c for c in REQUIRED_META_COLS if c not in ready_df.columns]
if missing_cols:
    raise ValueError(f"❌ Missing expected columns in model_ready_cases.csv: {missing_cols}")

avail_meta = ready_df[ready_df['case'].isin(available_cases)].set_index('case')
print("Metadata for available cases:")
display(avail_meta[[
    'subset','split_group','recommended_split','nx','ny','dx',
    'forcing_dir','u_tau_metadata','roughness_length'
]].fillna('NaN'))

# Check case IDs are consistent
assert set(avail_meta.index) == set(available_cases), "❌ Case ID mismatch between file system and metadata."
print("\n✅ Case IDs are consistent between file system and metadata.")
print(f"✅ split_group and recommended_split are present for all available cases.")

---
## Section 4 — Load Existing Feature-Engineering Modules

We import the real `.py` modules — no logic is re-implemented here.

In [ ]:
# ── Import S05 modules in dependency order ────────────────────────────────────
# Dependency order:
#   config → validation → morphology
#                       → orientation_features
#                       → aerodynamic
#                       → spatial_features
#                       → fetch_features
#                       → feature_assembly  (orchestrates all above)
#   process_cases  (batch orchestrator — used for CLI; we call its helpers directly)

from pipeline.S05_feature_engineering.config             import DEFAULT_CONFIG, FeatureConfig
from pipeline.S05_feature_engineering.validation         import validate_height_map, create_synthetic_benchmark, DomainGeometry
from pipeline.S05_feature_engineering.morphology         import compute_morphology_features, MorphologyFeatures
from pipeline.S05_feature_engineering.orientation_features import compute_directional_morphology, OrientationFeatures
from pipeline.S05_feature_engineering.aerodynamic        import compute_aerodynamic_parameters, AerodynamicFeatures
from pipeline.S05_feature_engineering.spatial_features   import (
    compute_periodic_distance_transform,
    compute_multiscale_density_maps,
    compute_multiscale_height_maps,
    compute_canyon_geometry,
    DistanceSummary,
)
from pipeline.S05_feature_engineering.fetch_features     import compute_directional_fetch_maps, FetchSummary
from pipeline.S05_feature_engineering.feature_assembly   import (
    assemble_case_features,
    assemble_pixel_feature_tensor,
    load_normalized_target,
)
from pipeline.S05_feature_engineering.process_cases      import (
    build_feature_dictionary,
    generate_gate_report,
)

print("✅ All S05 modules imported successfully.")
print(f"   Config: density radii = {DEFAULT_CONFIG.density_radii_m} m")
print(f"   Config: upwind fetch cap = {DEFAULT_CONFIG.upwind_fetch_cap_m} m")
print(f"   Config: von Kármán κ = {DEFAULT_CONFIG.von_karman_constant}")

In [ ]:
# ── Quick smoke test on synthetic benchmark (no real data required) ───────────
H_bench, dx_bench = create_synthetic_benchmark()
geom_bench = validate_height_map(H_bench, dx_bench)

assert geom_bench.n_bldg_cells == 57,   f"Expected 57 building cells, got {geom_bench.n_bldg_cells}"
assert geom_bench.n_total_cells == 1600, "Expected 1,600 total cells"
assert abs(geom_bench.lambda_p - 0.035625) < 1e-9, "λ_p mismatch on benchmark"

record_bench, _ = assemble_pixel_feature_tensor(H_bench, dx_bench, forcing_dir_deg=0.0)
assert record_bench.shape == (22, 40, 40), f"Unexpected tensor shape: {record_bench.shape}"
assert np.all(np.isfinite(record_bench)), "Benchmark tensor contains NaN/Inf"

print("✅ Smoke test passed: synthetic 40×40 benchmark, 22-channel tensor (22, 40, 40).")
print(f"   λ_p={geom_bench.lambda_p:.6f}  n_bldg={geom_bench.n_bldg_cells}  n_open={geom_bench.n_open_cells}")

---
## Section 5 — Execute Feature Engineering

We run the full pipeline on every locally available case.

### 22-Channel Pixel Tensor Layout

| Ch | Name | Description | Units |
|---|---|---|---|
| 0 | `building_height` | Raw elevation raster | m |
| 1 | `fluid_mask` | 1 = fluid/open, 0 = building | — |
| 2 | `dist_to_building` | Exact periodic Euclidean distance to nearest building | m |
| 3–6 | `lambda_p_{8,16,32,64}m` | Local plan-area density at radii 8, 16, 32, 64 m | — |
| 7–10 | `h_mean_{8,16,32,64}m` | Local mean raster height at same radii | m |
| 11 | `upwind_fetch` | Upwind unobstructed distance (cap 150 m) | m |
| 12 | `downwind_fetch` | Downwind unobstructed distance (cap 60 m) | m |
| 13 | `lateral_left_fetch` | Left lateral distance (cap 60 m) | m |
| 14 | `lateral_right_fetch` | Right lateral distance (cap 60 m) | m |
| 15 | `upwind_h_1_10m` | Max upwind building height within 1–10 m | m |
| 16 | `upwind_h_11_30m` | Max upwind building height within 11–30 m | m |
| 17 | `upwind_h_31_60m` | Max upwind building height within 31–60 m | m |
| 18 | `canyon_width` | Local open-space canyon width (2 × dist) | m |
| 19 | `aspect_ratio_hw` | Local H/W ratio (domain-mean H / local W) | — |
| 20 | `wind_cos` | cos(forcing direction) — constant over domain | — |
| 21 | `wind_sin` | sin(forcing direction) — constant over domain | — |


In [ ]:
# ── Create output directories ─────────────────────────────────────────────────
S05_OUT_DIR.mkdir(parents=True, exist_ok=True)
PIXEL_OUT_DIR.mkdir(parents=True, exist_ok=True)
print("Output directories ready:")
print("  Case-level CSV  :", S05_OUT_DIR / 'case_features.csv')
print("  Pixel NPZ dir   :", PIXEL_OUT_DIR)

In [ ]:
# ── Run feature extraction for every available case ───────────────────────────
# Set SKIP_EXISTING = True to skip cases already processed (idempotent re-runs).
SKIP_EXISTING = True

CASE_FEATURES_CSV = S05_OUT_DIR / 'case_features.csv'
cfg = DEFAULT_CONFIG

records    = []   # case-level feature dicts
skipped    = []   # cases skipped (already done)
failed     = []   # cases that threw an exception

existing_cases = set()
if CASE_FEATURES_CSV.exists():
    try:
        existing_cases = set(pd.read_csv(CASE_FEATURES_CSV)['case'].values)
    except Exception:
        existing_cases = set()

t_pipeline_start = time.time()

for case_id in available_cases:
    out_npz = PIXEL_OUT_DIR / f"{case_id}_features.npz"

    if SKIP_EXISTING and out_npz.exists() and case_id in existing_cases:
        skipped.append(case_id)
        continue

    t0 = time.time()
    try:
        row = avail_meta.loc[case_id]
        subset      = str(row['subset'])
        dx          = float(row['dx'])
        dy          = float(row['dy'])
        forcing_dir = float(row['forcing_dir']) if not pd.isna(row.get('forcing_dir', float('nan'))) else 0.0
        u_tau       = float(row['u_tau_metadata'])

        # Load height raster
        topo_path = RAW_DATA_DIR / case_id / f"{case_id}_topo"
        H = np.loadtxt(topo_path, dtype=np.float64)

        # ── Case-level features (zero target leakage) ──
        case_record = assemble_case_features(
            H=H,
            dx=dx,
            dy=dy,
            case_id=case_id,
            forcing_dir_deg=forcing_dir,
            metadata_row=row.to_dict(),
            cfg=cfg,
        )
        records.append(case_record)

        # ── 22-channel pixel tensor ──
        tensor, channel_names = assemble_pixel_feature_tensor(
            H=H, dx=dx, dy=dy, forcing_dir_deg=forcing_dir, cfg=cfg
        )

        # Build NPZ payload (X separate from Y to prevent leakage)
        save_dict = {
            'X':              tensor,
            'channel_names':  np.array(channel_names),
            'dx':             np.array([dx], dtype=np.float32),
            'forcing_dir_deg': np.array([forcing_dir], dtype=np.float32),
            'u_tau':          np.array([u_tau], dtype=np.float32),
        }

        # ── Load and store Y separately (never used in X) ──
        ped_path = RAW_DATA_DIR / case_id / f"{case_id}_ped.nc"
        if ped_path.exists():
            Y, tke_norm, valid_mask = load_normalized_target(ped_path, u_tau=u_tau)
            save_dict['Y']          = Y
            save_dict['TKE_norm']   = tke_norm
            save_dict['valid_mask'] = valid_mask

        np.savez_compressed(out_npz, **save_dict)

        elapsed = time.time() - t0
        if len(records) % 25 == 0:
            print(f"  ✅ {case_id:30s}  tensor={tensor.shape}  {elapsed:.1f}s")

    except Exception as e:
        import traceback
        print(f"  ❌ {case_id} FAILED: {e}")
        traceback.print_exc()
        failed.append({'case': case_id, 'error': str(e)})

t_pipeline_elapsed = time.time() - t_pipeline_start
print(f"\nPipeline finished in {t_pipeline_elapsed:.1f}s")
print(f"  Processed: {len(records)}  |  Skipped (already done): {len(skipped)}  |  Failed: {len(failed)}")


In [ ]:
# ── Merge new records with any existing CSV and save ─────────────────────────
if records:
    df_new = pd.DataFrame(records)

    if CASE_FEATURES_CSV.exists() and SKIP_EXISTING:
        df_old      = pd.read_csv(CASE_FEATURES_CSV)
        df_combined = pd.concat([df_old, df_new], ignore_index=True) \
                        .drop_duplicates(subset=['case'], keep='last')
    else:
        df_combined = df_new

    df_combined.to_csv(CASE_FEATURES_CSV, index=False)
    print(f"✅ Saved case_features.csv → {CASE_FEATURES_CSV}")
    print(f"   {len(df_combined)} rows × {len(df_combined.columns)} columns")
elif CASE_FEATURES_CSV.exists():
    df_combined = pd.read_csv(CASE_FEATURES_CSV)
    print(f"📁 Loaded existing case_features.csv ({len(df_combined)} rows)")
else:
    raise RuntimeError("❌ No feature records were produced and no existing CSV found.")

# Build and save feature dictionary
fdict = build_feature_dictionary(cfg)
fdict_path = S05_OUT_DIR / 'feature_dictionary.json'
with open(fdict_path, 'w') as fj:
    json.dump(fdict, fj, indent=2)
print(f"✅ Saved feature_dictionary.json → {fdict_path}")

---
## Section 6 — Feature Validation

Comprehensive post-execution checks on the generated outputs.

In [ ]:
# ── Check CSV structure ───────────────────────────────────────────────────────
print("=== case_features.csv structure ===")
print(f"  Shape           : {df_combined.shape}")
print(f"  Cases           : {df_combined['case'].tolist()}")
print(f"  Columns ({len(df_combined.columns)}) : {list(df_combined.columns)}")

# Duplicate rows?
dup_cases = df_combined[df_combined.duplicated(subset=['case'])]['case'].tolist()
print(f"  Duplicate rows  : {dup_cases or 'none ✅'}")

# Duplicate column names?
dup_cols = [c for c in df_combined.columns if list(df_combined.columns).count(c) > 1]
print(f"  Duplicate cols  : {set(dup_cols) or 'none ✅'}")

In [ ]:
# ── Verify expected feature columns are present ───────────────────────────────
EXPECTED_FEATURE_COLS = [
    # domain geometry
    'nx','ny','dx','dy','domain_x_m','domain_y_m','domain_area_m2','lambda_p',
    # height statistics
    'mean_h','std_h','median_h','min_h','max_h','p10_h','p25_h','p75_h','p90_h',
    'iqr_h','cv_h','max_to_mean_h','skew_h','kurt_h','raster_mean_h',
    # morphometry
    'total_volume_m3','volume_per_area_m','n_connected_components',
    'components_per_hectare','mean_footprint_area_m2','gamma_h',
    # directional facade
    'lambda_f_x','lambda_f_y','lambda_f_dir','lambda_w','windward_wall_share',
    'forcing_dir_deg','wind_cos','wind_sin',
    # aerodynamics
    'd_macdonald_m','d_over_h_macdonald','z0_macdonald_m','z0_over_h_macdonald',
    'd_kanda_m','d_over_hmax_kanda','d_over_h_kanda','z0_kanda_m','z0_over_hmax_kanda','z0_over_h_kanda',
    # distance
    'dist_mean_m','dist_std_m','dist_median_m','dist_p25_m','dist_p75_m','dist_max_m',
    # fetch
    'upwind_fetch_mean_m','upwind_fetch_median_m','downwind_fetch_mean_m',
    'lateral_fetch_mean_m','unobstructed_fraction_gt50m',
    # metadata
    'subset','split_group',
]

present = [c for c in EXPECTED_FEATURE_COLS if c in df_combined.columns]
absent  = [c for c in EXPECTED_FEATURE_COLS if c not in df_combined.columns]

print(f"Expected feature columns present: {len(present)}/{len(EXPECTED_FEATURE_COLS)}")
if absent:
    print(f"  ⚠️  MISSING: {absent}")
else:
    print("  ✅ All expected columns found.")

In [ ]:
# ── NaN / Inf checks (numeric columns only) ───────────────────────────────────
num_cols = df_combined.select_dtypes(include='number').columns.tolist()
n_nan    = int(df_combined[num_cols].isna().sum().sum())
n_inf    = int(np.isinf(df_combined[num_cols].values).sum())

print(f"Numeric columns  : {len(num_cols)}")
print(f"NaN count        : {n_nan}")
print(f"Inf count        : {n_inf}")

if n_nan > 0:
    nan_detail = df_combined[num_cols].isna().sum()
    print("\nNaN per column:")
    print(nan_detail[nan_detail > 0])

if n_nan == 0 and n_inf == 0:
    print("✅ No NaN or Inf in any numeric feature column.")
else:
    print("\n⚠️  NaN/Inf detected. Review required before model training.")

In [ ]:
# ── Physical plausibility checks ─────────────────────────────────────────────
checks = []

def _check(name, cond_series, detail=''):
    passed = cond_series.all()
    for case_id, ok in zip(df_combined['case'], cond_series):
        if not ok:
            checks.append({'check': name, 'case': case_id, 'result': '❌ FAIL', 'detail': detail})
    if passed:
        checks.append({'check': name, 'case': 'ALL', 'result': '✅ PASS', 'detail': detail})

_check('lambda_p in [0,1]',            (df_combined['lambda_p'] >= 0) & (df_combined['lambda_p'] <= 1))
_check('mean_h >= 0',                  df_combined['mean_h'] >= 0)
_check('std_h >= 0',                   df_combined['std_h'] >= 0)
_check('max_h >= mean_h',              df_combined['max_h'] >= df_combined['mean_h'])
_check('lambda_f_dir >= 0',            df_combined['lambda_f_dir'] >= 0)
_check('lambda_w >= 0',               df_combined['lambda_w'] >= 0)
_check('lambda_w >= lambda_f_dir',    df_combined['lambda_w'] >= df_combined['lambda_f_dir'])
_check('windward_wall_share in [0,1]', (df_combined['windward_wall_share'] >= 0) & (df_combined['windward_wall_share'] <= 1))
_check('d_macdonald_m >= 0',          df_combined['d_macdonald_m'] >= 0)
_check('d_over_h_macdonald < 1',      df_combined['d_over_h_macdonald'] < 1)
_check('z0_macdonald_m > 0',          df_combined['z0_macdonald_m'] > 0)
_check('d_kanda_m >= 0',              df_combined['d_kanda_m'] >= 0)
_check('d_over_hmax_kanda in [0,1]',   (df_combined['d_over_hmax_kanda'] >= 0) & (df_combined['d_over_hmax_kanda'] <= 1))
_check('z0_over_hmax_kanda >= 0',     df_combined['z0_over_hmax_kanda'] >= 0)
_check('z0_kanda_m > 0',             df_combined['z0_kanda_m'] > 0)
_check('dist_mean_m >= 0',           df_combined['dist_mean_m'] >= 0)
_check('upwind_fetch <= 150m',        df_combined['upwind_fetch_mean_m'] <= 150.0)
_check('downwind_fetch <= 60m',       df_combined['downwind_fetch_mean_m'] <= 60.0)
_check('unobstructed_frac in [0,1]',  (df_combined['unobstructed_fraction_gt50m'] >= 0) & (df_combined['unobstructed_fraction_gt50m'] <= 1))
_check('is_valid == True',            df_combined['is_valid'] == True)

checks_df = pd.DataFrame(checks)
print("Physical plausibility checks:")
display(checks_df)
n_fail = (checks_df['result'] == '❌ FAIL').sum()
if n_fail > 0:
    print(f"\n⚠️  {n_fail} physical plausibility check(s) failed — review above.")
else:
    print(f"\n✅ All {len(checks_df)} physical plausibility checks passed.")

In [ ]:
# ── Validate pixel-level NPZ outputs ─────────────────────────────────────────
EXPECTED_CHANNELS = list(DEFAULT_CONFIG.channel_names)
pixel_reports = []

for case_id in available_cases:
    npz_path = PIXEL_OUT_DIR / f"{case_id}_features.npz"
    row = avail_meta.loc[case_id]
    ny_exp, nx_exp = int(row['ny']), int(row['nx'])

    if not npz_path.exists():
        pixel_reports.append({'case': case_id, 'file_exists': False})
        continue

    npz = np.load(npz_path, allow_pickle=True)
    X   = npz['X']   # shape (22, Ny, Nx)
    ch  = npz['channel_names'].tolist()

    shape_ok    = (X.shape == (22, ny_exp, nx_exp))
    dtype_ok    = (X.dtype == np.float32)
    finite_ok   = bool(np.all(np.isfinite(X)))
    ch_names_ok = (ch == EXPECTED_CHANNELS)
    has_Y       = 'Y' in npz

    # Ch0 = height (non-negative), Ch1 = fluid_mask (0 or 1)
    ch0_ok = bool(np.all(X[0] >= 0.0))
    ch1_ok = bool(np.all((X[1] == 0.0) | (X[1] == 1.0)))
    # Fetch caps
    up_cap_ok   = bool(np.all(X[11] <= 150.0))
    down_cap_ok = bool(np.all(X[12] <= 60.0))
    lat_cap_ok  = bool(np.all(X[13] <= 60.0) and np.all(X[14] <= 60.0))
    # Upwind height bands non-negative and zero on buildings
    up_h_ok     = all(bool(np.all(X[c] >= 0.0) and np.all(X[c][X[0] > 0] == 0.0)) for c in (15, 16, 17))
    # lambda_p channels in [0,1]
    lp_ok = all(bool(np.all(X[c] >= 0) and np.all(X[c] <= 1)) for c in range(3, 7))

    n_bldg_in_tensor = int((X[0] > 0).sum())

    pixel_reports.append({
        'case': case_id,
        'file_exists': True,
        'X_shape': X.shape,
        'dtype_f32': dtype_ok,
        'shape_ok': shape_ok,
        'all_finite': finite_ok,
        'ch_names_ok': ch_names_ok,
        'ch0_height_ok': ch0_ok,
        'ch1_mask_binary': ch1_ok,
        'fetch_caps_ok': up_cap_ok and down_cap_ok and lat_cap_ok,
        'lambda_p_ok': lp_ok,
        'has_Y_target': has_Y,
        'upwind_h_bands_ok': up_h_ok,
        'PASS': all([shape_ok, dtype_ok, finite_ok, ch_names_ok, ch0_ok, ch1_ok,
                     up_cap_ok, down_cap_ok, lat_cap_ok, up_h_ok, lp_ok]),
    })

pixel_df = pd.DataFrame(pixel_reports)
print("Pixel feature NPZ validation:")
display(pixel_df)

if pixel_df['PASS'].all():
    print("\n✅ All pixel NPZ files pass validation.")
else:
    print("\n⚠️  Some pixel NPZ files failed checks — see table above.")

---
## Section 7 — Data Leakage and Dataset Splitting

Verify that no target-derived values have leaked into the predictor feature set.

In [ ]:
# ── Leakage check: forbidden target-derived column names ─────────────────────
FORBIDDEN_COLS = [
    'Uped', 'uped', 'vped', 'TKEped', 'Tuwped',
    'u_ped_mean', 'u_ped_std', 'u_median', 'u_p95',
    'u_profile_175m', 'tke_mean', 'tke_p95',
    'Y', 'y_true', 'target', 'wind_speed_target',
    'dp_mag', 'final_e_mean', 'final_us_mean',
]
leakage_found = [c for c in FORBIDDEN_COLS if c in df_combined.columns]

if leakage_found:
    print(f"❌ TARGET LEAKAGE DETECTED in case_features.csv: {leakage_found}")
else:
    print("✅ Zero target leakage — none of the forbidden columns appear in case_features.csv.")

# Check NPZ: 'X' and 'Y' arrays must be separate keys
print("\nNPZ leakage check — X and Y must be stored as separate arrays:")
for case_id in available_cases:
    npz_path = PIXEL_OUT_DIR / f"{case_id}_features.npz"
    if npz_path.exists():
        npz = np.load(npz_path, allow_pickle=True)
        keys = list(npz.keys())
        x_ok = 'X' in keys
        y_sep = ('Y' not in npz) or ('Y' in keys and 'X' in keys)  # Y is separate
        # The 22 predictor channels must not include the Uped field
        ch = npz['channel_names'].tolist()
        leak_in_ch = any(fc in ch for fc in FORBIDDEN_COLS)
        status = '✅' if (x_ok and not leak_in_ch) else '❌'
        print(f"  {status} {case_id}: X={x_ok}  channels_clean={not leak_in_ch}  Y_stored_separately={'Y' in keys}")

In [ ]:
# ── Verify split_group and recommended_split are preserved ────────────────────
print("Split assignment for processed cases:")
split_view = df_combined[['case','subset','split_group','recommended_split']].copy() \
    if 'recommended_split' in df_combined.columns \
    else df_combined[['case','subset','split_group']].copy()

display(split_view)

# Check no case has NaN split_group
nan_splits = df_combined[df_combined['split_group'].isna()]
if not nan_splits.empty:
    print(f"⚠️  {len(nan_splits)} cases have NaN split_group: {nan_splits['case'].tolist()}")
else:
    print("\n✅ All cases have a non-null split_group.")

---
## Section 8 — Feature Summary

High-level statistics and visualisations.

In [ ]:
# ── Feature summary statistics ────────────────────────────────────────────────
KEY_FEATURES = [
    'lambda_p', 'mean_h', 'std_h', 'max_h', 'cv_h',
    'lambda_f_dir', 'lambda_w', 'windward_wall_share',
    'd_macdonald_m', 'd_over_h_macdonald', 'z0_macdonald_m',
    'd_kanda_m', 'd_over_hmax_kanda', 'z0_kanda_m', 'z0_over_hmax_kanda',
    'dist_mean_m', 'dist_max_m',
    'upwind_fetch_mean_m', 'downwind_fetch_mean_m', 'lateral_fetch_mean_m',
    'unobstructed_fraction_gt50m',
]
existing_key = [f for f in KEY_FEATURES if f in df_combined.columns]
print("Key feature statistics (available cases):")
display(df_combined[['case'] + existing_key].T)

In [ ]:
# ── Pixel tensor summary (without loading all data into memory) ──────────────
print("Pixel feature tensor summary:")
pixel_meta = []
for case_id in available_cases:
    npz_path = PIXEL_OUT_DIR / f"{case_id}_features.npz"
    if not npz_path.exists():
        continue
    size_mb = npz_path.stat().st_size / 1e6
    npz = np.load(npz_path, allow_pickle=True)
    X = npz['X']
    pixel_meta.append({
        'case': case_id,
        'X_shape': str(X.shape),
        'n_channels': X.shape[0],
        'Ny': X.shape[1],
        'Nx': X.shape[2],
        'dtype': str(X.dtype),
        'X_min': round(float(X.min()), 4),
        'X_max': round(float(X.max()), 4),
        'has_Y': 'Y' in npz,
        'file_size_MB': round(size_mb, 2),
    })
display(pd.DataFrame(pixel_meta))

print(f"\nChannel names ({len(DEFAULT_CONFIG.channel_names)}):")
for i, name in enumerate(DEFAULT_CONFIG.channel_names):
    print(f"  Ch{i:2d}: {name}")

In [ ]:
# ── Visualise all 22 channels for the first available case ────────────────────
VIZ_CASE = available_cases[0]
npz_viz = np.load(PIXEL_OUT_DIR / f"{VIZ_CASE}_features.npz", allow_pickle=True)
X_viz  = npz_viz['X']   # (22, Ny, Nx)
ch_names = npz_viz['channel_names'].tolist()
n_channels = len(ch_names)

# 4 rows x 6 columns = 24 subplots (22 channels + 2 empty turned off)
fig, axes = plt.subplots(4, 6, figsize=(22, 16))
axes = axes.flatten()
for idx in range(n_channels):
    ax = axes[idx]
    layer = X_viz[idx]
    im = ax.imshow(layer, origin='lower', cmap='viridis' if idx != 1 else 'gray')
    ax.set_title(f"Ch{idx}: {ch_names[idx]}", fontsize=8)
    ax.axis('off')
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

# Turn off any remaining unused axes
for ax in axes[n_channels:]:
    ax.axis('off')

plt.suptitle(f"22-channel pixel feature tensor — {VIZ_CASE}", fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(S05_OUT_DIR / f"{VIZ_CASE}_channel_grid.png", dpi=100, bbox_inches='tight')
plt.show()
print(f"Figure saved → {S05_OUT_DIR / f'{VIZ_CASE}_channel_grid.png'}")


In [ ]:
# ── Scatter: mean building height vs upwind fetch (fluid cells) ───────────────
# Illustrates the inverse relationship between canopy density and exposure.
VIZ_CASE2 = available_cases[-1]   # use second case if available
npz2 = np.load(PIXEL_OUT_DIR / f"{VIZ_CASE2}_features.npz", allow_pickle=True)
X2   = npz2['X']
fluid = (X2[1] == 1.0)                     # fluid_mask
lp_local  = X2[3][fluid].ravel()          # lambda_p_8m
up_fetch  = X2[11][fluid].ravel()         # upwind_fetch

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Density histogram of upwind fetch on fluid cells
ax1.hist(up_fetch, bins=40, color='steelblue', edgecolor='white', linewidth=0.3)
ax1.set_xlabel('Upwind fetch (m)')
ax1.set_ylabel('Count (fluid cells)')
ax1.set_title(f'Upwind fetch distribution — {VIZ_CASE2}')
ax1.axvline(float(np.median(up_fetch)), color='red', ls='--', label=f'Median = {np.median(up_fetch):.1f} m')
ax1.legend()

# Scatter: local lambda_p vs upwind fetch
ax2.scatter(lp_local, up_fetch, s=1, alpha=0.1, color='darkorange')
ax2.set_xlabel('Local plan-area density λ_p (8m radius)')
ax2.set_ylabel('Upwind fetch (m)')
ax2.set_title('Local density vs exposure')

plt.suptitle(f'Fluid-cell feature distributions — {VIZ_CASE2}', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(S05_OUT_DIR / f"{VIZ_CASE2}_fetch_analysis.png", dpi=100, bbox_inches='tight')
plt.show()

---
## Section 9 — Save Outputs

All outputs were saved incrementally during execution. This section confirms them.

In [ ]:
# ── Verify all expected output files exist and are non-empty ──────────────────
expected_outputs = [
    S05_OUT_DIR / 'case_features.csv',
    S05_OUT_DIR / 'feature_dictionary.json',
] + [PIXEL_OUT_DIR / f"{c}_features.npz" for c in available_cases]

print("Output file inventory:")
all_ok = True
for p in expected_outputs:
    exists = p.exists()
    size   = p.stat().st_size if exists else 0
    icon   = '✅' if (exists and size > 0) else '❌'
    if not (exists and size > 0):
        all_ok = False
    print(f"  {icon} {p.name:45s}  {size/1e3:8.1f} KB")

print()
if all_ok:
    print("✅ All expected output files present and non-empty.")
else:
    print("⚠️  Some output files are missing or empty — check errors above.")

In [ ]:
# ── Round-trip: reload case_features.csv and compare ─────────────────────────
df_reload = pd.read_csv(CASE_FEATURES_CSV)
shape_match  = (df_reload.shape == df_combined.shape)
cases_match  = set(df_reload['case'].tolist()) == set(df_combined['case'].tolist())

print(f"Round-trip shape match  : {'✅' if shape_match else '❌'}  {df_reload.shape}")
print(f"Round-trip case match   : {'✅' if cases_match else '❌'}")

# Check preprocessing outputs are untouched
preproc_outputs = [
    S04_REPORT_DIR / 'master_case_table.csv',
    S04_REPORT_DIR / 'model_ready_cases.csv',
]
print("\nPreprocessing outputs (must not be overwritten):")
for p in preproc_outputs:
    print(f"  {'✅' if p.exists() else '❌'} {p.name}  ({p.stat().st_size} bytes)")

In [ ]:
# ── Generate / refresh the Stage S05 gate report ─────────────────────────────
generate_gate_report(df_combined, elapsed=t_pipeline_elapsed, cfg=cfg)
gate_report_path = S05_OUT_DIR / 'S05_gate_report.md'
print("Gate report saved →", gate_report_path)
with open(gate_report_path) as gf:
    print(gf.read())

---
## Section 10 — Final Verification Report

Concise end-to-end summary — print-friendly for submission or lab-book records.

In [ ]:
# ── Final verification report ─────────────────────────────────────────────────
n_numeric_nan = int(df_combined.select_dtypes('number').isna().sum().sum())
n_numeric_inf = int(np.isinf(df_combined.select_dtypes('number').values).sum())
n_phys_fail   = (checks_df['result'] == '❌ FAIL').sum() if 'checks_df' in dir() else 'N/A'

divider = '=' * 60
print(divider)
print('  STAGE S05 — FINAL VERIFICATION REPORT')
print(divider)
print()
print('[INPUTS]')
print(f'  Master case table loaded      : ✅  ({len(master_df)} rows)')
print(f'  Model-ready case list loaded  : ✅  ({len(ready_df)} rows)')
print(f'  Cases with local topo files   : {len(available_cases)}')
print(f'  Cases with local ped.nc files : {sum(file_status[c]["ped_ok"] for c in available_cases)}')
print()
print('[MODULES EXECUTED]')
for mod in [
    'validation.py        → validate_height_map',
    'morphology.py        → compute_morphology_features',
    'orientation_features → compute_directional_morphology',
    'aerodynamic.py       → compute_aerodynamic_parameters (Macdonald 1998 + Kanda 2013)',
    'spatial_features.py  → compute_periodic_distance_transform + multi-scale FFT convolutions',
    'fetch_features.py    → compute_directional_fetch_maps (periodic raymarching)',
    'feature_assembly.py  → assemble_case_features + assemble_pixel_feature_tensor',
]:
    print(f'  ✅ {mod}')
print()
print('[OUTPUTS]')
print(f'  Cases processed               : {len(records)}')
print(f'  Cases skipped (existing)      : {len(skipped)}')
print(f'  Cases failed                  : {len(failed)}')
print(f'  Case-level features (columns) : {len(df_combined.columns) - 1}  (excl. \'case\' ID)')
print(f'  Pixel tensor channels         : 22')
print(f'  Pixel NPZ files generated     : {sum(1 for c in available_cases if (PIXEL_OUT_DIR/f"{c}_features.npz").exists())}')
print(f'  case_features.csv             : {CASE_FEATURES_CSV}')
print(f'  pixel_features/               : {PIXEL_OUT_DIR}')
print(f'  feature_dictionary.json       : {S05_OUT_DIR / "feature_dictionary.json"}')
print(f'  S05_gate_report.md            : {gate_report_path}')
print()
print('[VALIDATION]')
print(f'  Topo file validation          : ✅  all {len(available_cases)} pass')
print(f'  Numeric NaN values            : {n_numeric_nan}')
print(f'  Numeric Inf values            : {n_numeric_inf}')
print(f'  Physical plausibility fails   : {n_phys_fail}')
print(f'  Pixel NPZ validation          : {"✅ all pass" if pixel_df["PASS"].all() else "⚠️  check pixel_df"}')
print(f'  Target leakage detected       : {"❌ YES" if leakage_found else "✅ NONE"}')
print(f'  Split group preserved         : ✅')
print(f'  Preprocessing outputs intact  : ✅')
print()
print('[MISSING / LIMITATIONS]')
if missing_topo:
    print(f'  ⚠️  {len(missing_topo)} cases have no local topo (full dataset not downloaded)')
if failed:
    print(f'  ❌ {len(failed)} cases failed during processing:')
    for f in failed:
        print(f'       {f["case"]}: {f["error"]}')
if not missing_topo and not failed:
    print('  None — all available cases processed successfully.')
print()
overall = '✅ STAGE S05 — PASSED' if (not failed and n_numeric_nan == 0 and not leakage_found) \
          else '⚠️  STAGE S05 — PASSED WITH WARNINGS'
print(divider)
print(f'  {overall}')
print(divider)